In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, silhouette_score
)
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")


In [4]:
# ─── 1. LOAD DATA ────────────────────────────────────────────────────────────

train = pd.read_csv("Train.csv")
test  = pd.read_csv("Test.csv")
sample_sub = pd.read_csv("sample_submission.csv")

In [5]:
print("=" * 60)
print("DATASET SELECTION")
print("=" * 60)
print("""
  sample_submission → Only 'ID' + 'Segmentation' (output template, no features)
  Test              → Features only, NO 'Segmentation' label (for prediction)
  Train             → Features + 'Segmentation' label ← USE THIS FOR ANALYSIS
""")
print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Sample sub  : {sample_sub.shape}\n")

DATASET SELECTION

  sample_submission → Only 'ID' + 'Segmentation' (output template, no features)
  Test              → Features only, NO 'Segmentation' label (for prediction)
  Train             → Features + 'Segmentation' label ← USE THIS FOR ANALYSIS

Train shape : (8068, 11)
Test shape  : (2627, 10)
Sample sub  : (2627, 2)



In [6]:
# ─── 2. EXPLORE ──────────────────────────────────────────────────────────────

print("=" * 60)
print("TRAIN DATASET OVERVIEW")
print("=" * 60)
print(train.head())
print("\nData Types:\n", train.dtypes)
print("\nMissing Values:\n", train.isnull().sum())
print("\nTarget Distribution:\n", train["Segmentation"].value_counts())
print("\nBasic Stats:\n", train.describe())

TRAIN DATASET OVERVIEW
       ID  Gender Ever_Married  Age Graduated     Profession  Work_Experience  \
0  462809    Male           No   22        No     Healthcare              1.0   
1  462643  Female          Yes   38       Yes       Engineer              NaN   
2  466315  Female          Yes   67       Yes       Engineer              1.0   
3  461735    Male          Yes   67       Yes         Lawyer              0.0   
4  462669  Female          Yes   40       Yes  Entertainment              NaN   

  Spending_Score  Family_Size  Var_1 Segmentation  
0            Low          4.0  Cat_4            D  
1        Average          3.0  Cat_4            A  
2            Low          1.0  Cat_6            B  
3           High          2.0  Cat_6            B  
4           High          6.0  Cat_6            A  

Data Types:
 ID                   int64
Gender              object
Ever_Married        object
Age                  int64
Graduated           object
Profession          object
Wo

In [7]:
# ─── 3. PREPROCESSING ────────────────────────────────────────────────────────

df = train.copy()

# Fill missing values
df["Ever_Married"].fillna(df["Ever_Married"].mode()[0], inplace=True)
df["Graduated"].fillna(df["Graduated"].mode()[0], inplace=True)
df["Profession"].fillna(df["Profession"].mode()[0], inplace=True)
df["Work_Experience"].fillna(df["Work_Experience"].median(), inplace=True)
df["Family_Size"].fillna(df["Family_Size"].median(), inplace=True)
df["Var_1"].fillna(df["Var_1"].mode()[0], inplace=True)


In [8]:
print("\nMissing values after imputation:\n", df.isnull().sum())



Missing values after imputation:
 ID                 0
Gender             0
Ever_Married       0
Age                0
Graduated          0
Profession         0
Work_Experience    0
Spending_Score     0
Family_Size        0
Var_1              0
Segmentation       0
dtype: int64


In [9]:
# Encode categoricals
le = LabelEncoder()
cat_cols = ["Gender", "Ever_Married", "Graduated", "Profession", "Spending_Score", "Var_1"]
for col in cat_cols:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))

In [10]:
df["Segmentation_enc"] = le.fit_transform(df["Segmentation"])

In [11]:
feature_cols = [c + "_enc" for c in cat_cols] + ["Age", "Work_Experience", "Family_Size"]
X = df[feature_cols]
y = df["Segmentation_enc"]

In [12]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [13]:
# ─── 4. EDA CHARTS (Plotly) ───────────────────────────────────────────────────

# 4a. Segment distribution
fig1 = px.pie(
    df, names="Segmentation",
    title="Customer Segment Distribution",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.write_html("chart_segment_distribution.html")
fig1.show()

In [14]:
# 4b. Age distribution by segment
fig2 = px.box(
    df, x="Segmentation", y="Age", color="Segmentation",
    title="Age Distribution by Segment",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig2.write_html("chart_age_by_segment.html")
fig2.show()

In [15]:
# 4c. Spending score by segment
fig3 = px.histogram(
    df, x="Spending_Score", color="Segmentation", barmode="group",
    title="Spending Score Distribution by Segment",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.write_html("chart_spending_by_segment.html")
fig3.show()


In [16]:
# 4d. Profession heatmap
prof_seg = df.groupby(["Profession", "Segmentation"]).size().reset_index(name="Count")
fig4 = px.density_heatmap(
    prof_seg, x="Profession", y="Segmentation", z="Count",
    title="Profession vs Segment Heatmap",
    color_continuous_scale="Blues"
)
fig4.write_html("chart_profession_heatmap.html")
fig4.show()

In [17]:
# 4e. Gender & Married breakdown
fig5 = px.sunburst(
    df, path=["Gender", "Ever_Married", "Segmentation"],
    title="Gender → Married → Segment Sunburst",
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig5.write_html("chart_sunburst.html")
fig5.show()

In [18]:
# 4f. Work Experience vs Age scatter
fig6 = px.scatter(
    df, x="Age", y="Work_Experience", color="Segmentation",
    size="Family_Size", hover_data=["Profession", "Spending_Score"],
    title="Age vs Work Experience (sized by Family Size)",
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig6.write_html("chart_scatter.html")
fig6.show()


In [19]:
# ─── 5. KMEANS CLUSTERING ────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("KMEANS — Elbow Method")
print("=" * 60)


KMEANS — Elbow Method


In [20]:
inertias = []
sil_scores = []
K_range = range(2, 10)

In [21]:
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))
    print(f"  k={k}  inertia={km.inertia_:.0f}  silhouette={sil_scores[-1]:.4f}")

  k=2  inertia=58040  silhouette=0.2018
  k=3  inertia=52367  silhouette=0.1717
  k=4  inertia=46687  silhouette=0.1859
  k=5  inertia=43200  silhouette=0.1858
  k=6  inertia=40632  silhouette=0.1831
  k=7  inertia=38488  silhouette=0.1774
  k=8  inertia=36766  silhouette=0.1709
  k=9  inertia=34935  silhouette=0.1767


In [22]:
fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(x=list(K_range), y=inertias, mode="lines+markers", name="Inertia"))
fig_elbow.update_layout(title="Elbow Curve for KMeans", xaxis_title="k", yaxis_title="Inertia")
fig_elbow.write_html("chart_elbow.html")
fig_elbow.show()

In [23]:
best_k = 4
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["KMeans_Cluster"] = km_final.fit_predict(X_scaled)

In [24]:
# PCA visualisation
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

In [25]:
fig_pca = px.scatter(
    df, x="PCA1", y="PCA2", color=df["KMeans_Cluster"].astype(str),
    title="KMeans Clusters (PCA 2D projection)",
    labels={"color": "Cluster"},
    color_discrete_sequence=px.colors.qualitative.Set1
)
fig_pca.write_html("chart_kmeans_pca.html")
fig_pca.show()

In [26]:
# ─── 6. SUPERVISED CLASSIFICATION ────────────────────────────────────────────

print("\n" + "=" * 60)
print("RANDOM FOREST CLASSIFIER")
print("=" * 60)


RANDOM FOREST CLASSIFIER


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [28]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

In [29]:
acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.4f}")
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["A", "B", "C", "D"]))


Accuracy: 0.4932

Classification Report:

              precision    recall  f1-score   support

           A       0.39      0.39      0.39       394
           B       0.37      0.35      0.36       372
           C       0.56      0.51      0.53       394
           D       0.62      0.69      0.65       454

    accuracy                           0.49      1614
   macro avg       0.48      0.48      0.48      1614
weighted avg       0.49      0.49      0.49      1614



In [30]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig_imp = px.bar(
    importances, orientation="h",
    title="Feature Importances (Random Forest)",
    labels={"value": "Importance", "index": "Feature"},
    color=importances.values,
    color_continuous_scale="Blues"
)
fig_imp.write_html("chart_feature_importance.html")
fig_imp.show()

In [31]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(
    cm, text_auto=True,
    x=["A", "B", "C", "D"], y=["A", "B", "C", "D"],
    color_continuous_scale="Blues",
    title="Confusion Matrix — Random Forest"
)
fig_cm.write_html("chart_confusion_matrix.html")
fig_cm.show()


In [32]:
# ─── 7. SEGMENT PROFILES ─────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("SEGMENT PROFILES")
print("=" * 60)


SEGMENT PROFILES


In [33]:
profile = df.groupby("Segmentation").agg(
    Count=("ID", "count"),
    Avg_Age=("Age", "mean"),
    Avg_Work_Exp=("Work_Experience", "mean"),
    Avg_Family_Size=("Family_Size", "mean"),
    Top_Profession=("Profession", lambda x: x.mode()[0]),
    Pct_Graduated=("Graduated", lambda x: (x == "Yes").mean() * 100),
    Pct_Married=("Ever_Married", lambda x: (x == "Yes").mean() * 100),
).reset_index()


In [34]:
print(profile.to_string(index=False))

Segmentation  Count   Avg_Age  Avg_Work_Exp  Avg_Family_Size Top_Profession  Pct_Graduated  Pct_Married
           A   1972 44.924949      2.690162         2.466531         Artist      63.032454    59.381339
           B   1858 48.200215      2.235737         2.703983         Artist      72.658773    74.219591
           C   1970 49.144162      2.143147         2.975127         Artist      82.335025    79.796954
           D   2268 33.390212      2.764991         3.216931     Healthcare      36.640212    29.144621


In [35]:
fig_radar_data = df.groupby("Segmentation")[["Age", "Work_Experience", "Family_Size"]].mean()
fig_radar = go.Figure()
for seg in fig_radar_data.index:
    fig_radar.add_trace(go.Scatterpolar(
        r=fig_radar_data.loc[seg].values.tolist() + [fig_radar_data.loc[seg].values[0]],
        theta=["Age", "Work Experience", "Family Size", "Age"],
        fill="toself", name=f"Segment {seg}"
    ))
fig_radar.update_layout(title="Segment Radar Chart (Numeric Features)")
fig_radar.write_html("chart_radar.html")
fig_radar.show()

In [36]:
# ─── 8. PREDICT TEST SET ─────────────────────────────────────────────────────

test_df = test.copy()
for col in cat_cols:
    test_df[col + "_enc"] = le.fit_transform(test_df[col].fillna(test_df[col].mode()[0]).astype(str))
for num_col in ["Age", "Work_Experience", "Family_Size"]:
    test_df[num_col].fillna(test_df[num_col].median(), inplace=True)

In [37]:
X_test_final = scaler.transform(test_df[feature_cols])
test_df["Segmentation"] = le.inverse_transform(rf.predict(X_test_final))

In [38]:
submission = test_df[["ID", "Segmentation"]]
submission.to_csv("my_submission.csv", index=False)
print("\nSubmission saved to my_submission.csv")
print(submission["Segmentation"].value_counts())


Submission saved to my_submission.csv
Segmentation
Cat_4    777
Cat_1    651
Cat_3    607
Cat_2    592
Name: count, dtype: int64


In [39]:
print("\n✅ All charts exported as HTML. Open them in any browser.")



✅ All charts exported as HTML. Open them in any browser.
